# Демонстрация библиотеки simintech-api

Python-библиотека управления SimInTech через COM API.

> **Внимание**: ячейки, помеченные `# COM`, требуют Windows с
> зарегистрированным COM-сервером (`mmain.exe /regserver`). В среде без COM
> они пропускаются автоматически. Ячейки layout/графиков работают везде.


In [ ]:
# Подключение библиотеки и COM-сервера
from simintech_api import COMClient, Project
import os

client = None
try:
    client = COMClient(silent_mode=True).connect()
    print("COM подключён (PID:", client.get_process_id(), ")")
except Exception as exc:
    print("COM недоступен — layout/графики работают, модели без COM не строятся.")
    print("Причина:", exc)


## Модель 1: Усилитель сигнала

Схема `Синусоида (1 Гц) → Усилитель (×2.5)`. Расчёт, чтение выхода.

In [ ]:
# COM: создание модели усилителя
if client:
    prj = Project.new(client)
    page = prj.get_main_page()
    src = page.create_block("Синусоида", 0, 0)
    src.set_property("Name", "SinSource")
    gain = page.create_block("Усилитель", 200, 0)
    gain.set_property("Name", "Gain")
    gain.set_property("a", 2.5)
    src.connect(gain)
    prj.save_xml("demo_amplifier.xprt")
    print("Модель 1 сохранена: demo_amplifier.xprt")
else:
    print("Модель 1 пропущена (нет COM)")


## Модель 2: ПИД-регулятор

Ступенька (уставка) → сумматор ошибки → ПИД-ветви → интегратор (объект),
обратная связь выхода на второй вход сумматора.

In [ ]:
# COM: ПИД-регулятор
if client:
    prj = Project.new(client)
    page = prj.get_main_page()
    ref = page.create_block("Ступенька", 0, -120)
    ref.set_property("Name", "Step"); ref.set_property("yk", 1.0)
    err = page.create_block("Сумматор", 180, -120)
    err.set_property("Name", "Sum"); err.set_property("a", [1.0, -1.0])
    kp = page.create_block("Усилитель", 360, -200); kp.set_property("a", 1.0)
    ki = page.create_block("Усилитель", 360, -120); ki.set_property("a", 0.5)
    kd = page.create_block("Усилитель", 360, -40); kd.set_property("a", 0.1)
    pid_sum = page.create_block("Сумматор", 520, -120)
    # У «Сумматора» по умолчанию 2 входа; более длинный `a` портов не добавит
    pid_sum.set_property("Name", "PID"); pid_sum.set_in_port_count(3)
    pid_sum.set_property("a", [1.0, 1.0, 1.0])
    plant = page.create_block("Интегратор", 700, -120)
    plant.set_property("Name", "Plant"); plant.set_property("k", 1.0)
    ref.connect(err, in_index=0)
    err.connect(kp); err.connect(ki); err.connect(kd)
    kp.connect(pid_sum, in_index=0); ki.connect(pid_sum, in_index=1)
    kd.connect(pid_sum, in_index=2); pid_sum.connect(plant)
    plant.connect(err, in_index=1)   # обратная связь
    prj.save_xml("demo_pid.xprt")
    print("Модель 2 сохранена: demo_pid.xprt")
else:
    print("Модель 2 пропущена (нет COM)")


## Модель 3: многокомпонентная система (layout + router)

12 блоков, 14 перекрёстных связей. `LayeredPlacer` расставляет блоки без
наложений, `AStarRouter` трассирует линии в обход блоков. **Работает без COM**.

In [ ]:
from simintech_api.layout import LayeredPlacer, AStarRouter, ObstacleGrid

spec = ['src1','src2','sum1','g1','g2','int1','sum2','g3','g4','int2','sum3','plot']
links = [('src1','sum1'),('src2','sum1'),('sum1','g1'),('sum1','g2'),('g1','int1'),
         ('g2','sum2'),('int1','sum2'),('g1','sum2'),('sum2','g3'),('sum2','g4'),
         ('g3','int2'),('int2','sum3'),('g4','sum3'),('sum3','plot')]
w, h = 60.0, 40.0

pos = LayeredPlacer().place(spec, links, sizes={k: (w, h) for k in spec})
print("Расстановка блоков завершена, блоков:", len(spec))


In [ ]:
# Таблица координат блоков
rows = [(bid, round(cx, 1), round(cy, 1)) for bid, (cx, cy) in pos.items()]
# Печать без pandas (универсально)
print(f"{'Блок':6s} {'x':>8s} {'y':>8s}")
for bid, cx, cy in rows:
    print(f"{bid:6s} {cx:8.1f} {cy:8.1f}")


In [ ]:
# Упрощённая визуализация: сетка блоков
x_vals = [pos[k][0] for k in spec]; y_vals = [pos[k][1] for k in spec]
print("Диапазон X:", min(x_vals), "-", max(x_vals), " Y:", min(y_vals), "-", max(y_vals))
print("Блоков:", len(spec), "Наложений:",
      sum(1 for i,a in enumerate(pos.values())
          for b in list(pos.values())[i+1:] if abs(a[0]-b[0])<40 and abs(a[1]-b[1])<40))


In [ ]:
# Трассировка линий через блоки (A*)
grid = ObstacleGrid(250, 150)
for bid, (cx, cy) in pos.items():
    grid.add_rect(cx - w/2, cy - h/2, w, h)
router = AStarRouter()
results = []
for src, dst in links:
    s, d = pos[src], pos[dst]
    p1 = (s[0] + w/2, s[1]); p2 = (d[0] - w/2, d[1])
    try:
        pts = router.route(p1, p2, grid, start_side=1, end_side=0)
        results.append((src, dst, pts))
    except Exception as exc:
        results.append((src, dst, f"НЕ НАЙДЕН: {exc}"))
print(f"Линий проложено: {sum(1 for _,_,p in results if isinstance(p, list))}/{len(links)}")
for src, dst, pts in results[:5]:
    print(f"  {src}->{dst}: {pts if isinstance(pts, list) else pts}")


## Вывод

- `LayeredPlacer` расставляет блоки без наложений (слои по направлению сигнала).
- `AStarRouter` прокладывает линии в обход блоков (ортогональный A*, штраф за повороты).
- Модели 1–2 (COM) строятся на Windows; модель 3 работает в любом окружении.

Документация: `docs/guide.md`, `docs/api.md`, `docs/algorithms.md`.
